import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from google.cloud import bigquery

load_dotenv(Path('..').resolve() / '.env')
sns.set_theme(style='whitegrid')

PROJECT_ID = os.environ["GCP_PROJECT_ID"]
client = bigquery.Client(project=PROJECT_ID)

def q(sql):
    return client.query(sql).to_dataframe()


In [ ]:
q(f"""SELECT activity_band, COUNT(*) AS users
    FROM `{PROJECT_ID}`.marts.mart_user_engagement
    GROUP BY 1 ORDER BY 2 DESC""")


q(f"""SELECT primary_genre, completion_rate, abandon_rate
    FROM `{PROJECT_ID}`.marts.mart_completion_rates_by_genre
    ORDER BY completion_rate DESC NULLS LAST LIMIT 15""")


In [ ]:
q(f"""SELECT r.user_id, r.rank, r.title_id, t.title_name,
          t.primary_genre, r.predicted_score
    FROM `{PROJECT_ID}`.marts.mart_user_recommendations r
    JOIN `{PROJECT_ID}`.marts.dim_title t
      ON r.title_id = t.title_business_key
   WHERE r.user_id = (
       SELECT MIN(user_id) FROM `{PROJECT_ID}`.marts.mart_user_recommendations)
   ORDER BY r.rank""")


m = q(f"SELECT * FROM `{PROJECT_ID}`.meta.experiment_metrics")
m.groupby(['condition','metric_name'])['metric_value'].agg(['mean','std','count']).reset_index()


In [ ]:
wide = m.pivot_table(
    index=['run_uuid','metric_name'], columns='condition',
    values='metric_value', aggfunc='first').reset_index()
rows = []
for metric in ['hit_rate_at_10','ndcg_at_10','coverage_at_10','rec_age_hours','credits_used']:
    sub = wide[wide['metric_name']==metric].dropna(
        subset=['BASELINE_NIGHTLY','INTERVENTION_HOURLY'])
    if len(sub) < 5: continue
    b = sub['BASELINE_NIGHTLY']; i = sub['INTERVENTION_HOURLY']
    diff = (i - b).values
    w_stat, p = stats.wilcoxon(i, b)
    d = diff.mean() / (diff.std(ddof=1)+1e-12)
    rows.append({'metric': metric, 'mean_diff': diff.mean(),
                 'p_value': p, 'cohens_d': d, 'significant': p < 0.05})
pd.DataFrame(rows)


## Top recommended titles for a sample user

In [ ]:
q('''SELECT r.user_id, r.rank, r.title_id, t.title_name, t.primary_genre, r.predicted_score
    FROM MARTS.MART_USER_RECOMMENDATIONS r
    JOIN MARTS.DIM_TITLE t ON r.title_id = t.title_business_key
   WHERE r.user_id = (SELECT MIN(user_id) FROM MARTS.MART_USER_RECOMMENDATIONS)
   ORDER BY r.rank''')

## Experiment: nightly vs hourly

In [ ]:
m = q('SELECT * FROM META.experiment_metrics')
m.groupby(['condition','metric_name'])['metric_value'].agg(['mean','std','count']).reset_index()

In [ ]:
wide = m.pivot_table(index=['run_uuid','metric_name'], columns='condition',
                    values='metric_value', aggfunc='first').reset_index()
rows = []
for metric in ['hit_rate_at_10','ndcg_at_10','coverage_at_10','rec_age_hours','credits_used']:
    sub = wide[wide['metric_name']==metric].dropna(subset=['BASELINE_NIGHTLY','INTERVENTION_HOURLY'])
    if len(sub) < 5: continue
    b = sub['BASELINE_NIGHTLY']; i = sub['INTERVENTION_HOURLY']
    diff = (i - b).values
    w_stat, p = stats.wilcoxon(i, b)
    d = diff.mean() / (diff.std(ddof=1)+1e-12)
    rows.append({'metric': metric, 'mean_diff': diff.mean(),
                 'p_value': p, 'cohens_d': d, 'significant': p < 0.05})
pd.DataFrame(rows)